# Coverage and Sparsity

A focused Step 4 notebook for the two data realities that drive the modeling plan: field coverage and tiny player samples.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

ROOT = Path.cwd().resolve()
if not (ROOT / 'outputs').exists():
    ROOT = ROOT.parent

df = pd.read_parquet(ROOT / 'outputs' / 'statsbomb' / 'penalties_statsbomb_clean.parquet')
df.shape

In [ ]:
shooter_counts = df.groupby(['shooter_id', 'shooter_name']).size().rename('n_pens').reset_index()
keeper_counts = df.dropna(subset=['keeper_id']).groupby(['keeper_id', 'keeper_name']).size().rename('n_faced').reset_index()

shooter_counts['n_pens'].describe(), keeper_counts['n_faced'].describe()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].hist(shooter_counts['n_pens'], bins=range(1, int(shooter_counts['n_pens'].max()) + 2), color='#4C78A8', edgecolor='white')
axes[0].set_title('Penalties per shooter')
axes[0].set_xlabel('Recorded penalties')
axes[0].set_ylabel('Shooters')

axes[1].hist(keeper_counts['n_faced'], bins=range(1, int(keeper_counts['n_faced'].max()) + 2), color='#F58518', edgecolor='white')
axes[1].set_title('Penalties faced per resolved keeper')
axes[1].set_xlabel('Recorded penalties faced')
axes[1].set_ylabel('Keepers')

fig.tight_layout()

In [ ]:
coverage_fields = ['shot_zone', 'keeper_id', 'keeper_dive_direction', 'gk_freeze_location_x', 'gk_freeze_location_y', 'shootout_kick_index']
coverage = pd.DataFrame({
    'field': coverage_fields,
    'non_null': [df[c].notna().sum() for c in coverage_fields],
    'coverage': [df[c].notna().mean() for c in coverage_fields],
})
coverage

In [ ]:
perfect_tiny = (
    df.groupby(['shooter_id', 'shooter_name'])['outcome_bin']
    .agg(n='size', goals='sum', conversion='mean')
    .reset_index()
    .query('n <= 2 and conversion == 1')
    .sort_values(['n', 'shooter_name'], ascending=[False, True])
)
perfect_tiny.head(20)

In [ ]:
# Regenerate the committed figures and memo.
%run ../reports/step4_eda.py